# 📅 2026-05-22 (금) ~ 2026-05-24 (일) 개발 노트
## 추천 엔진 v5 + Phase 1 완료

## 🎯 기간 목표
- [x] 추천 엔진 v5 재설계 (점수 씹창 문제 해결)
- [x] 핵심 테스트 24개 작성 + 통과
- [x] DB 백업 cron 등록
- [x] Sentry 모니터링 연동
- [x] Discord 알람 연동
- [x] UserAction Django 모델 추가
- [x] 프로젝트 정리 (venv, .gitignore, scripts 폴더)

---

## 🔴 핵심 문제 발견 및 해결

### 문제 1: 매치 점수 전부 52점으로 동일
```
증상: Metal Gear Solid, Spider-Man, Witcher 3, Fallout 전부 52점
원인 1: _normalize_scores()가 0.5 + norm * 0.5 스케일
         → 최하위도 50점, 차이 거의 없음
원인 2: 4,190개 전체 49차원 벡터 코사인 비교
         → 벡터 방향이 비슷해서 변별력 없음
원인 3: 기준 게임이 결과에 포함됨
         → '림월드 같은 게임' 검색에 림월드 나옴
```

### 해결: 추천 엔진 v5 전면 재설계

#### 1. 점수 시스템 (절대 점수)
```python
# 기존: min-max 상대 정규화 (다 비슷하게 나옴)
scaled = 0.5 + norm * 0.5  # 50~100 범위, 변별력 없음

# v5: sigmoid 변환 절대 점수
기준 게임 = 100점 앵커 (결과에서 제외, 별도 표시)
나머지   = 0~99점 (절대 초과 불가)

계산:
  지표 유사도(60%) = 가중치 유클리드(67%) + 코사인(33%)
  임베딩(40%) = pgvector 코사인
  → sigmoid 변환 → 0~94점
  + gem 보너스 최대 +5점
  = 최종 0~99점
```

#### 2. 4단계 가중치 시스템
```python
W_PRIMARY    = 5.0x  # 관련 지표 (3~5개)
W_SECONDARY  = 2.0x  # 연관 지표 (5~8개)
W_NEUTRAL    = 0.5x  # 공통 중요 지표
W_IRRELEVANT = 0.1x  # 나머지 (0이 아닌 약하게 반영)

# 변별력
관련 5개 × 5.0 = 25
무관 35개 × 0.1 = 3.5
→ 7배 변별력 차이

# 기존 6개 제한 완전 철회 → 전체 49개 지표 사용
# 무관 지표도 0.1x로 약하게 반영 (완전 0 아님)
```

#### 3. 의도별 가중치 그룹 (12개)
```python
mood_cozy:         primary=[cozy_factor, time_pressure, grind_factor]
mood_horror:       primary=[horror_factor, gore_level, dark_fantasy_vibe]
feature_strategy:  primary=[strategic_depth, management_complexity, learning_curve]
feature_roguelike: primary=[rng_dependency, replay_value, learning_curve]
feature_management: primary=[management_complexity, strategic_depth, freedom_level]
# + mood_dark, mood_narrative, feature_narrative,
#   feature_coop, feature_action, feature_puzzle, feature_exploration

# query_hint로 동적 적용
{"app_id": 294100, "count": 5, "query_hint": "림월드 같은 경영 게임"}
→ feature_management 가중치 적용
```

#### 4. 기준 게임 제외 + 100점 앵커
```python
# by-game 추천에서 기준 게임 결과 제외
.where(Game.app_id != app_id)  # 기준 게임 제외

# 기준 게임은 100점 앵커로 별도 반환
reference_game: target_game.name  # 응답에 포함
```

#### 5. score_breakdown 응답 구조
```json
{
  "metric_score": 78.5,
  "embedding_score": 82.1,
  "gem_bonus": 3.2,
  "final_score": 84.0
}
```

---

## 🧪 핵심 테스트 24개 작성

### P0 테스트 (16개) — 배포 차단 기준
```
TestCostGuardBlocksOverLimit (6개)
  - 일일 한도 초과 차단
  - 시간당 스파이크 차단
  - Redis 장애 시 fail-open
  - 설정값 sanity check

TestNullMetricsDontCrash (6개)
  - ZERO/GLOBAL_MEAN/GENRE_MEAN 폴백 정책
  - ALL-NULL 게임도 크래시 없음
  - NaN/inf 발생 없음

TestRecommendationReturnsValidResults (4개)
  - 점수 0~99 범위
  - 앵커 100 상수
  - 완벽 매치도 99 초과 불가
  - score_breakdown 필드 구조
```

### P1 테스트 (8개) — 품질 기준
```
TestRateLimitEnforced (4개)
  - 설정 존재, 형식, 범위, 익명≤인증

TestWeightingDifferentiatesScores (4개)
  - 가중치 10배+ 차이
  - mood_cozy 티어 배정 정확성
  - 유사 게임 > 다른 게임 점수
  - preferences 항상 primary 가중치
```

### 실행 결과
```bash
pytest tests/test_core.py -v -m p0  # 16 passed
pytest tests/test_core.py -v -m p1  # 8 passed
pytest tests/test_core.py -v        # 24 passed, 0 failed
```

---

## 🛠 인프라 작업

### DB 백업 자동화
```bash
# scripts/backup/backup_db.sh
# 기능: pg_dump → gzip → /backups/ 저장 → 7일 초과 삭제 → Discord 알람

# Windows Task Scheduler 등록
PowerShell -ExecutionPolicy Bypass -File scripts\backup\setup_task_scheduler.ps1
# → 매일 새벽 3:00 AM 자동 실행

# 수동 테스트 결과
✅ 백업 성공: backups/db_20260524_215239.sql.gz (39M)
```

### Sentry 모니터링
```python
# main.py에 init_sentry() 추가
sentry_sdk.init(
    dsn=settings.SENTRY_DSN,
    integrations=[FastApiIntegration(), SqlalchemyIntegration(), RedisIntegration()],
    traces_sample_rate=1.0,
)

# config.py에 추가
SENTRY_DSN: str = ""
SENTRY_ENV: str = "development"
SENTRY_TRACES_SAMPLE_RATE: float = 1.0

# docker-compose.yml에 추가
- SENTRY_DSN=${SENTRY_DSN}
- SENTRY_ENV=${SENTRY_ENV:-development}

# 테스트: curl http://127.0.0.1:8000/ops/sentry-test
# → Sentry Issues 탭에서 에러 수신 확인
```

### Discord 알람 연동
```python
# .env에 추가
DISCORD_WEBHOOK_URL=https://discord.com/api/webhooks/...

# docker-compose.yml에 추가
- DISCORD_WEBHOOK_URL=${DISCORD_WEBHOOK_URL}

# 알람 발생 조건
OpenAI $30 초과 → ⚠️ 경고
OpenAI $50 초과 → 🚨 차단
시간당 $5 초과  → 🚨 차단
DB 백업 성공/실패 → ✅/🚨
```

### UserAction Django 모델 (Phase 1.5)
```python
# django_core/apps/users/models.py에 추가
class UserAction(models.Model):
    user       = ForeignKey(CustomUser, null=True)  # 비로그인도 추적
    session_id = CharField(max_length=64)            # 비로그인 추적용
    app_id     = IntegerField(null=True)             # 대상 게임
    action_type = CharField(choices=ActionType)      # 행동 종류
    context    = JSONField()                         # 모든 컨텍스트 저장
    created_at = DateTimeField(auto_now_add=True)

# 인덱스 3개
idx_ua_user_time    # 유저별 시간순
idx_ua_app_action   # 게임별 행동
idx_ua_session_time # 세션별 행동

# 마이그레이션
docker exec hidden_gem_django python manage.py makemigrations users
docker exec hidden_gem_django python manage.py migrate
# → Applying users.0002... OK
```

---

## 🗂 프로젝트 정리

### scripts/ 폴더 구조화
```
scripts/
├── backup/      # DB 백업 + 스케줄러
├── pipeline/    # GPT 배치 + 데이터 적재
├── images/      # Steam 헤더 이미지
├── utils/       # 유틸리티
└── README.md
```

### requirements 분리
```
requirements.txt      # 프로덕션 (Docker 이미지)
requirements-dev.txt  # 개발/테스트 (로컬 venv)
  -r requirements.txt
  pytest==8.1.1
  pytest-asyncio==0.23.6
  pytest-mock==3.14.0
```

### venv 정리
```
# 정리 전
C:/Hidden-Gem-project/.venv/           # 루트
C:/Hidden-Gem-project/fastapi_app/.venv/ # 중복

# 정리 후
C:/Hidden-Gem-project/.venv/           # 하나만 유지
```

### .gitignore 정비
```
# fastapi_app/.gitignore 신규 생성
.venv/
__pycache__/
*.pyc
.pytest_cache/
.env
*.log

# 루트 .gitignore에 backups/ 추가
# (DB 백업 파일 39MB가 실수로 커밋됨 → git rm --cached backups/ 로 제거)
```

### PRD v3.1 → v3.2 업데이트
```
완료로 이동: 추천 엔진 마스킹/점수/테스트
신규 추가: 추천 엔진 v5 설계, 테스트 전략, UserWeightStrategy
```

---

## 🚨 트러블슈팅

### 1. ImportError: No module named 'services'
```
원인: pytest가 fastapi_app/ 루트를 Python path로 인식 못함
해결: pytest.ini에 pythonpath = . 추가
      conftest.py에 sys.path.insert(0, os.path.join(os.path.dirname(__file__), "..")) 추가
```

### 2. CostGuard mock 실패 (AttributeError: _get_current_cost)
```
원인: cost_guard.py가 _get_redis() → r.get() 직접 호출 구조
      _get_current_cost() 메서드 없음
해결: _get_redis()를 mock하여 Redis get() 반환값 제어
      mock_redis.get = async def(key): return str(daily_cost)
```

### 3. Sentry DSN 전달 안 됨
```
원인 1: config.py에 SENTRY_DSN 필드 없음
         → getattr(settings, 'SENTRY_DSN', None)이 None 반환
원인 2: docker-compose.yml에 SENTRY_DSN 환경변수 미전달
해결: config.py에 SENTRY_DSN: str = "" 추가
      docker-compose.yml fastapi environment에 - SENTRY_DSN=${SENTRY_DSN} 추가
```

### 4. DB 백업 파일 39MB Git 커밋
```
원인: backups/ 폴더가 .gitignore에 없었음
해결: git rm -r --cached backups/
      .gitignore에 backups/, *.sql.gz 추가
      → 히스토리에는 남아있음 (나중에 BFG로 제거 가능)
```

### 5. asyncio_mode STRICT 경고
```
원인: conftest.py의 event_loop fixture가 deprecated
해결: pytest.ini에 asyncio_mode = auto 설정
      conftest.py에서 event_loop fixture 제거
```

---

## 💡 인사이트

- **코사인 유사도 함정**: 49차원 전체 비교 시 방향이 비슷해서 변별력 사라짐
  → 가중치 차등으로 관련 지표를 5x 부스트해야 실제 차이 나옴
- **절대 점수 vs 상대 정규화**: 상대 정규화는 보기 좋지만 의미 없는 숫자
  → sigmoid 변환 절대 점수 + 앵커 100점이 신뢰도 높음
- **테스트 우선순위**: P0(비용/크래시)가 P1(품질)보다 먼저
  → CostGuard 테스트 없으면 $1000 청구서 맞을 수 있음
- **취향 학습 오버엔지니어링 방지**: 유저 0명 상태에서 학습 로직 만들면 검증 불가
  → Phase 1.5는 데이터 수집 인프라만, 로직은 Phase 2+

---

## 📋 다음 할 일 (Phase 1.5 → Phase 2)

### 즉시 (프론트 UI 개선)
- [ ] 점수 표시 개선 (0~99 + 앵커 100 표현)
- [ ] 게임 카드 매치 점수 변별력 표현
- [ ] 에러 상태 UI + 재시도 버튼
- [ ] 게임 상세 → 비슷한 게임 섹션
- [ ] 무한 스크롤

### Phase 1.5
- [ ] POST /taste/action 엔드포인트 (Phase 2 회원 시스템과 연계)

### Phase 2
- [ ] Google OAuth2 + Steam OpenID
- [ ] 스와이프 온보딩
- [ ] 취향 DNA 카드
- [ ] 지표 3개 추가 (accessibility_options, speedrun_potential, world_density)

### Phase 3
- [ ] Vercel + Railway 배포
- [ ] 취향 학습 (유저 100명+, 행동 1만 건+)